<a href="https://colab.research.google.com/github/mbaker21231/MicroII-Sandbox/blob/main/Hopenhayn2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Code to simulate Hopenhayn model

A first requirement is a utility function to make a continuous distribution into a grid. Here it is:

In [1]:
#Packages

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

I think it is a good idea to define all the parameters we are using in one place, and basically keep track of them on the basis of whether they are globals or locals, and experimental values.

In [14]:
N   =   100
MU  =    -.25
RHO =     .85
SIGMA =   .40

In [2]:
def tauchen(N, mu, rho, sigma, n_std=4):
    z = np.linspace(mu - n_std * sigma / np.sqrt(1 - rho**2),
                     mu + n_std * sigma / np.sqrt(1 - rho**2), N)
    step = (z[1] - z[0])
    P = np.zeros((N, N))

    for j in range(N):
        for k in range(N):
            if k == 0:
                P[j, k] = norm.cdf((z[k] - rho * z[j] + step / 2) / sigma)
            elif k == N-1:
                P[j, k] = 1 - norm.cdf((z[k] - rho * z[j] - step / 2) / sigma)
            else:
                P[j, k] = (norm.cdf((z[k] - rho * z[j] + step / 2) / sigma) -
                           norm.cdf((z[k] - rho * z[j] - step / 2) / sigma))

    return z, P


Note that we can also use this distribution to recover a cumulative unconditional distribution, which is useful for initial productivity draws:

In [3]:
def init_dist(N, mu, rho, sigma, n_std=4):

    tauch = tauchen(N, mu, rho, sigma, n_std)
    probs = np.sum(tauch[1], axis=0)/np.sum(tauch[1])
    vals = tauch[0]

    return vals, probs

## Aspects of the model

Each firm has a flow profit function of the form:
$$
\pi(z) = \tilde z n^\alpha - Wn
$$

where $\tilde z$ is the (exponentiated) skill level $z$, $\tilde z=e^z$. Flow profits are acheived by choosing $n$, labor, to maximize th above:

$$
 \alpha \tilde z n^{\alpha -1}-W \quad \rightarrow\quad n^*(z,w) = \left(\frac{\alpha \tilde z}{W}\right)^\frac{1}{1-\alpha}
$$

Here is a function that returns, for a given wage and skill level, profits and labor demand:

In [7]:
def prof_lab(z, W):

  z_tilde = np.exp(z)
  n_sta   = ( alpha * z_tilde / W)**(1/(1-alpha))
  profs   = z_tilde*n_sta**alpha - W*alpha

  return profs, n_sta

## Present value of a firm

The following bit of code essentially iterates the value function, taking into account that the firm's valuation changes as a result of possible changes in $z$, the skill level of the firm.

In [10]:
def val_fun(z, p, W, max_iter=3000, tol=1e-10, noisy=False):

  v = np.zeros((len(z), 1))

  for i in range(max_iter):

    profs = prof_lab(z, W)[0]
    profs = np.reshape(profs, (len(z), 1))
    vnew = np.maximum( 0, profs - phi_c + (1-beta)* p @ v)
    print(i, abs(vnew-v))
    if np.max(abs(vnew-v)<tol):
      break

  if noisy:
    print("Iterations: ", i)
    print("Maximum value: ", np.max(vnew))
    print("Average value: ", np.mnea(vnew))
    print("Minimum value: ", np.min(vnew))

  return vnew

## Computing an equilibrium wage

Let's first take a stab at computing an equilibrium wage. Intuitively, we want the wage to be such that the supply of labor is equal to the total demand for labor. Where firms that do not produce exit the market.

In [13]:
zhat, Phat = tauchen(N, mu, rho, sigma)


SyntaxError: invalid syntax (<ipython-input-13-2a010c79b4c0>, line 1)